# 03 - Faster R-CNN Detection

Notebook huấn luyện và kiểm tra Faster R-CNN để phát hiện vùng tổn thương da trong ảnh ISIC.

Pipeline: Image + Mask → Bounding Box → Faster R-CNN → Bounding Box + Confidence.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

## 1. Configuration

In [ ]:
IMAGE_DIR = Path("../../data/images/train")
MASK_DIR = Path("../../data/masks/train")
RESULT_DIR = Path("../../results/detection")

RESULT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 2
NUM_EPOCHS = 5
BATCH_SIZE = 2
LEARNING_RATE = 0.005

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)

## 2. Create Bounding Box from Segmentation Mask

Bounding Box được tạo từ các pixel lesion trong segmentation mask.

Format: [xmin, ymin, xmax, ymax].

In [ ]:
def mask_to_bbox(mask):
    mask = np.asarray(mask)
    ys, xs = np.where(mask > 0)

    if len(xs) == 0:
        return None

    xmin = float(xs.min())
    ymin = float(ys.min())
    xmax = float(xs.max())
    ymax = float(ys.max())

    return [xmin, ymin, xmax, ymax]


def find_image_mask_pairs():
    extensions = {".jpg", ".jpeg", ".png"}

    images = sorted([
        p for p in IMAGE_DIR.rglob("*")
        if p.is_file() and p.suffix.lower() in extensions
    ])

    pairs = []

    for image_path in images:
        mask_path = MASK_DIR / f"{image_path.stem}.png"

        if mask_path.exists():
            pairs.append((image_path, mask_path))

    return pairs


pairs = find_image_mask_pairs()

print("Valid image-mask pairs:", len(pairs))

## 3. Dataset

In [ ]:
class ISICDetectionDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        mask_array = np.array(mask)
        bbox = mask_to_bbox(mask_array)

        image_tensor = TF.to_tensor(image)

        if bbox is None:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.tensor([bbox], dtype=torch.float32)
            labels = torch.tensor([1], dtype=torch.int64)

            area = torch.tensor([
                max(0, (bbox[2] - bbox[0]) * (bbox[3] - bbox[1]))
            ], dtype=torch.float32)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([index]),
            "area": area,
            "iscrowd": torch.zeros(
                (len(labels),),
                dtype=torch.int64
            )
        }

        return image_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


dataset = ISICDetectionDataset(pairs)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

print("Dataset size:", len(dataset))

## 4. Visualize Ground Truth Bounding Box

In [ ]:
if pairs:
    image_path, mask_path = pairs[0]

    image = Image.open(image_path).convert("RGB")
    mask = Image.open(mask_path).convert("L")

    bbox = mask_to_bbox(mask)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(image)

    if bbox is not None:
        xmin, ymin, xmax, ymax = bbox

        rectangle = plt.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            fill=False,
            linewidth=2
        )

        ax.add_patch(rectangle)

    ax.set_title(
        f"Ground Truth Bounding Box - {image_path.stem}"
    )
    ax.axis("off")
    plt.show()
else:
    print("Không tìm thấy image-mask pair.")

## 5. Build Faster R-CNN Model

In [ ]:
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

model = fasterrcnn_resnet50_fpn(
    weights=weights
)

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    NUM_CLASSES
)

model = model.to(DEVICE)

print("Model: Faster R-CNN ResNet-50 FPN")
print("Classes:", NUM_CLASSES)

## 6. Training

Faster R-CNN sử dụng các loss thành phần cho classification, bounding box regression và RPN.

In [ ]:
params = [
    parameter for parameter in model.parameters()
    if parameter.requires_grad
]

optimizer = torch.optim.SGD(
    params,
    lr=LEARNING_RATE,
    momentum=0.9,
    weight_decay=0.0005
)

loss_history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0

    for images, targets in loader:
        images = [
            image.to(DEVICE)
            for image in images
        ]

        targets = [
            {
                key: value.to(DEVICE)
                for key, value in target.items()
            }
            for target in targets
        ]

        loss_dict = model(images, targets)
        total_loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        epoch_loss += total_loss.item()

    mean_loss = epoch_loss / max(1, len(loader))
    loss_history.append(mean_loss)

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} - "
        f"Loss: {mean_loss:.4f}"
    )

model_path = RESULT_DIR / "faster_rcnn_notebook.pth"
torch.save(model.state_dict(), model_path)

print("Model saved:", model_path)

## 7. Training Loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    range(1, len(loss_history) + 1),
    loss_history,
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("Total Loss")
plt.title("Faster R-CNN Training Loss")
plt.grid(True)
plt.show()

## 8. Inference

In [ ]:
model.eval()

if pairs:
    image_path, mask_path = pairs[0]

    image = Image.open(image_path).convert("RGB")
    image_tensor = TF.to_tensor(image).to(DEVICE)

    with torch.no_grad():
        prediction = model([image_tensor])[0]

    boxes = prediction["boxes"].cpu().numpy()
    scores = prediction["scores"].cpu().numpy()
    labels = prediction["labels"].cpu().numpy()

    print("Number of predictions:", len(boxes))

    if len(scores) > 0:
        print("Top confidence:", float(scores[0]))
        print("Top bounding box:", boxes[0].tolist())
else:
    print("Không có dữ liệu để inference.")

## 9. Visualize Detection Result

In [ ]:
CONFIDENCE_THRESHOLD = 0.5

if pairs:
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image)

    for box, score in zip(boxes, scores):
        if score < CONFIDENCE_THRESHOLD:
            continue

        xmin, ymin, xmax, ymax = box

        rectangle = plt.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            fill=False,
            linewidth=2
        )

        ax.add_patch(rectangle)
        ax.text(
            xmin,
            ymin,
            f"Lesion {score:.2f}"
        )

    ax.set_title("Faster R-CNN Detection Result")
    ax.axis("off")
    plt.show()
else:
    print("Không có kết quả detection để hiển thị.")

## 10. Detection Summary

Faster R-CNN có nhiệm vụ xác định vị trí tổn thương bằng Bounding Box và Confidence Score.

Bounding Box được sử dụng ở bước tiếp theo để crop ROI trước khi đưa vào U-Net segmentation.

In [ ]:
print("=" * 60)
print("FASTER R-CNN SUMMARY")
print("=" * 60)
print("Task       : Lesion Detection")
print("Model      : Faster R-CNN ResNet-50 FPN")
print("Classes    : Background + Lesion")
print("Epochs     :", NUM_EPOCHS)
print("Batch size :", BATCH_SIZE)
print("Device     :", DEVICE)
print("Confidence :", CONFIDENCE_THRESHOLD)
print("Next step  : ROI Crop -> U-Net Segmentation")
print("=" * 60)